# Borrower-Level Probability of Default Stress Testing

## Objective

The objective of this notebook is to translate macroeconomic stress scenarios into borrower-level stressed probabilities of default (PD).

Rather than applying a uniform stress multiplier across all borrowers, this notebook incorporates borrower-specific vulnerability characteristics. Borrowers with weaker credit quality are assumed to be more sensitive to adverse macroeconomic conditions than financially stronger borrowers.

The borrower vulnerability score is calculated using selected financial characteristics including:

- Debt-to-Income Ratio
- Credit Utilization
- Employment Stability
- Internal Credit Grade

The vulnerability score is combined with the macroeconomic stress multipliers derived in Notebook 07 to produce scenario-specific stressed PDs.

In [85]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [86]:
DATA = "../outputs/loan_pd_model.csv"
df = pd.read_csv(DATA)

In [87]:
import joblib

model = joblib.load("pd_model.pkl")

In [88]:
model_features = [
    "grade_grouped",
    "loan_to_credit_limit",
    "purpose_grouped",
    "debt_to_income",
    "log_loan_amount",
    "verified_income",
    "log_available_credit"
]

X = df[model_features]

df["Base_PD"] = model.predict_proba(X)[:,1]

In [89]:
risk_variables = [
    "loan_to_credit_limit",
    "debt_to_income",
    "log_loan_amount",
    "verified_income",
    "log_available_credit"
]

In [90]:
from sklearn.preprocessing import MinMaxScaler

risk = df[risk_variables].copy()

scaler = MinMaxScaler()

risk_scaled = pd.DataFrame(
    scaler.fit_transform(risk),
    columns=risk_variables,
    index=df.index
)

In [91]:
coef = pd.Series(
    np.abs(model.coef_[0]),
    index=model_features
)

In [92]:
weights = coef[risk_variables]

In [93]:
weights = weights / weights.sum()

weights

loan_to_credit_limit    0.140970
debt_to_income          0.025452
log_loan_amount         0.165218
verified_income         0.609169
log_available_credit    0.059190
dtype: float64

In [94]:
df["Borrower_Risk_Index"] = 0

In [95]:
for var in risk_variables:

    df["Borrower_Risk_Index"] += (

        risk_scaled[var]

        * weights[var]

    )

In [96]:
df["Borrower_Risk_Index"].describe()

count    10000.000000
mean         0.370827
std          0.238623
min          0.028145
25%          0.171377
50%          0.324862
75%          0.386013
max          0.831905
Name: Borrower_Risk_Index, dtype: float64

In [97]:
mean_score = df["Borrower_Risk_Index"].mean()

df["Borrower_Risk_Multiplier"] = (

    df["Borrower_Risk_Index"]

    / mean_score

)

In [98]:
df["Borrower_Risk_Multiplier"].describe()

count    10000.000000
mean         1.000000
std          0.643488
min          0.075899
25%          0.462147
50%          0.876047
75%          1.040952
max          2.243374
Name: Borrower_Risk_Multiplier, dtype: float64

In [99]:
scenario = {

    "Baseline":1.0,

    "Moderate Stress (GFC)":1.574797,

    "Pandemic Stress (COVID)":1.864833	

}

In [100]:
df["Stress_Downside"] = (

    scenario["Moderate Stress (GFC)"]

    * df["Borrower_Risk_Multiplier"]

)

df["Stress_Severe"] = (

    scenario["Pandemic Stress (COVID)"]

    * df["Borrower_Risk_Multiplier"]

)

In [101]:
X = df[model_features]

df["Base_PD"] = model.predict_proba(X)[:,1]

In [102]:
df["PD_Baseline"] = df["Base_PD"]

df["PD_Downside"] = np.clip(

    df["Base_PD"]

    * df["Stress_Downside"],

    0,

    1

)

df["PD_Severe"] = np.clip(

    df["Base_PD"]

    * df["Stress_Severe"],

    0,

    1

)

In [104]:
loan_level_pd = "../outputs/loan_level_pd.csv"
df.to_csv(loan_level_pd, index=False)

In [ ]:
summary = pd.DataFrame({

    "Scenario":[

        "Baseline",

        "Moderate Stress (GFC)",

        "Pandemic Stress (COVID)"

    ],

    "Average_PD":[

        df["PD_Baseline"].mean(),

        df["PD_Downside"].mean(),

        df["PD_Severe"].mean()

    ]

})

summary

,Scenario,Average_PD
0,Baseline,0.007188
1,Moderate Stress (GFC),0.014669
2,Pandemic Stress (COVID),0.017353
